# How to stream full state of your graph

LangGraph supports multiple streaming modes. The main ones are:

- `values`: This streaming mode streams back values of the graph. This is the **full state of the graph** after each node is called.
- `updates`: This streaming mode streams back updates to the graph. This is the **update to the state of the graph** after each node is called.

This guide covers `stream_mode="values"`.

## Setup

We'll be using a simple ReAct agent for this guide.

In [ ]:
# DEPENDENCY: pip install --quiet -U langchain langchain-openai langchain-community langchain-text-splitters langgraph langgraph-prebuilt langgraph-checkpoint-sqlite langsmith ddgs
# (packages should be pre-installed in venv before running this script)

In [ ]:
import getpass
import os
import warnings

from openai import OpenAI

# Suppress deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = os.environ.get("OPENAI_API_KEY")
    os.environ["OPENAI_API_KEY"] = secret_key

In [ ]:
from typing import Literal
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.runnables import ConfigurableField
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent


@tool
def get_weather(city: Literal["nyc", "sf"]):
    """Use this to get weather information."""
    if city == "nyc":
        return "It might be cloudy in nyc"
    elif city == "sf":
        return "It's always sunny in sf"
    else:
        raise AssertionError("Unknown city")


tools = [get_weather]

model = ChatOpenAI(temperature=0)
graph = create_react_agent(model, tools)

## Stream values

In [ ]:
inputs = {"messages": [("human", "what's the weather in sf")]}
for chunk in graph.stream(inputs, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

If we want to just get the final result, we can use the same method and just keep track of the last value we received

In [ ]:
inputs = {"messages": [("human", "what's the weather in sf")]}
for chunk in graph.stream(inputs, stream_mode="values"):
    final_result = chunk

In [6]:
final_result

{'messages': [HumanMessage(content="what's the weather in sf", id='02a2b402-3244-4169-a5f9-5a08da201505'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_dRCfMizOUkFhekAzsWR98Ggz', 'function': {'arguments': '{"city":"sf"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 58, 'total_tokens': 72, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-192a3483-3bc4-4ab3-842b-ca97f738295d-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'sf'}, 'id': 'call_dRCfMizOUkFhekAzsWR98Ggz', 'type': 'tool_call'}], usage_metadata={'input_tokens': 58, 'output_tokens': 14, 'total_tokens': 72}),
  ToolMessage(con

In [7]:
final_result["messages"][-1].pretty_print()

================================== Ai Message ==================================

The weather in San Francisco is always sunny!
